# 12 — Voice and Multimodal Agents

**Mode:** Live  
**Level:** Advanced  
**Estimated time:** 45 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

A real OpenAI media pipeline: committed WAV fixture → transcription → Agent response → speech synthesis → generated WAV → transcription verification. You will validate audio structure, listen to both files, inspect transcripts, and send a committed image as multimodal input.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(12, 'Voice and Multimodal Agents')


## Prerequisites and setup

**Course prerequisites:** Complete `course-09-model-runtime`.

**Execution requirements:** Live OpenAI. Set `OPENAI_API_KEY`, `PRAVAL_OPENAI_MODEL`, `PRAVAL_OPENAI_TRANSCRIPTION_MODEL`, `PRAVAL_OPENAI_TTS_MODEL`, and `PRAVAL_OPENAI_TTS_VOICE`. This protected lesson makes real STT/TTS calls.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Transcribe a real WAV fixture with provider STT.
- Pass the transcript through a real Agent.
- Synthesize and validate a real WAV reply, then transcribe it again.
- Send image content through the same provider-neutral Agent API.


## Mental model

Praval's voice APIs are request-based: upload audio for transcription, make an Agent request, and request synthesized speech. Multimodal input is represented as typed ContentParts. A **realtime session** is different: it keeps a persistent connection open for continuous event and audio exchange. Realtime sessions are not part of Praval 0.8.


In [ ]:
show_flow(
('Input WAV', 'real speech', 'human'),
('STT', 'text transcript', 'provider'),
('Agent', 'bounded reply', 'agent'),
('TTS + STT', 'audible verification', 'provider'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Prepare live media configuration

The helper reads real model names and creates a temporary output directory. WAV inspection uses Python's standard library, independent of provider metadata.


In [ ]:
import base64
import gzip
import tempfile
import wave

from IPython.display import Audio, Image, display
from praval import Agent, ContentPart

values = require_env(
    "OPENAI_API_KEY", "PRAVAL_OPENAI_MODEL",
    "PRAVAL_OPENAI_TRANSCRIPTION_MODEL", "PRAVAL_OPENAI_TTS_MODEL",
    "PRAVAL_OPENAI_TTS_VOICE",
)
workspace = tempfile.TemporaryDirectory(prefix="praval-voice-notebook-")
output = Path(workspace.name)


def wav_metadata(path):
    with wave.open(str(path), "rb") as stream:
        frames, rate = stream.getnframes(), stream.getframerate()
        return {
            "channels": stream.getnchannels(), "sample_width": stream.getsampwidth(),
            "sample_rate": rate, "frames": frames,
            "duration_seconds": round(frames / rate, 3),
        }


def normalized_words(value):
    clean = "".join(char.lower() if char.isalnum() else " " for char in value)
    return set(clean.split())


### What just happened?

The configuration names every paid model explicitly, and media validation does not trust a filename extension alone.

### Why this matters

Voice certification should verify the actual container and samples produced by the API.


### Load and listen to the committed WAV fixture

The fixture is compressed and base64 encoded for reliable source distribution. After decoding, its WAV header and sample count are inspected.


In [ ]:
voice_fixture = find_example_asset(
    "certification/assets/voice_input.wav.gz.base64"
)
input_audio = gzip.decompress(base64.b64decode(voice_fixture.read_text()))
input_path = output / "voice-input.wav"
input_path.write_bytes(input_audio)
input_info = wav_metadata(input_path)
assert input_info["frames"] > 0 and input_info["sample_rate"] > 0
assert input_info["duration_seconds"] > 0
stage("audio", "fixture loaded", f"{input_info['duration_seconds']} seconds")
display(Audio(filename=str(input_path)))
show_json(input_info, "Input WAV")


### What just happened?

A real, non-empty WAV is now available for provider transcription and human playback.

### Why this matters

Committed media fixtures make live runs comparable without replacing the external API.


### Transcribe real speech

The configured transcription model receives the WAV file. Assertions normalize words and tolerate punctuation while requiring the intended phrase.


In [ ]:
agent = Agent(
    "course-voice-agent", provider="openai",
    model=values["PRAVAL_OPENAI_MODEL"],
    config={"temperature": 0, "max_output_tokens": 64, "timeout": 60},
)
transcript = agent.transcribe(
    input_path, model=values["PRAVAL_OPENAI_TRANSCRIPTION_MODEL"],
    language="en", timeout=60,
)
required = {"praval", "speech", "transcription", "synthesis"}
assert required <= normalized_words(transcript)
stage("OpenAI STT", "transcribed", transcript)
show_json({"transcript": transcript}, "Real input transcription")


### What just happened?

The transcript came from the external STT service and contains the key fixture terms.

### Why this matters

Semantic keyword checks are stable across harmless punctuation and capitalization changes.


### Pass the transcript through the Agent

The text model receives the actual transcript and is instructed to return one bounded phrase suitable for round-trip speech verification.


In [ ]:
response = agent.generate(
    "The user said: " + transcript
    + " Reply with exactly: Praval voice round trip succeeded."
)
expected_reply = {"praval", "voice", "succeeded"}
assert expected_reply <= normalized_words(response.content)
stage("agent", "response generated", response.content)
show_json(
    {"provider": response.provider, "model": response.model,
     "reply": response.content,
     "usage": response.usage.model_dump() if response.usage else None},
    "Agent response",
)


### What just happened?

The Agent completed a real provider request using the transcript as input and returned a semantically constrained reply.

### Why this matters

Keeping the reply bounded lowers TTS cost and makes round-trip validation robust.


### Generate and validate real speech

`speak()` requests WAV output from the configured TTS model. The returned bytes are written, parsed, and made audible in the notebook.


In [ ]:
reply_audio = agent.speak(
    response.content, model=values["PRAVAL_OPENAI_TTS_MODEL"],
    voice=values["PRAVAL_OPENAI_TTS_VOICE"],
    response_format="wav", timeout=60,
)
reply_path = output / "voice-reply.wav"
reply_path.write_bytes(reply_audio)
reply_info = wav_metadata(reply_path)
assert reply_info["frames"] > 0 and reply_info["sample_width"] > 0
assert reply_info["duration_seconds"] > 0
stage("OpenAI TTS", "speech generated", f"{reply_info['duration_seconds']} seconds")
display(Audio(filename=str(reply_path)))
show_json(reply_info, "Generated WAV")


### What just happened?

The TTS service returned a valid WAV container with nonzero samples, duration, and sample rate.

### Why this matters

A successful HTTP response is not enough; generated media must be structurally usable.


### Transcribe the generated speech again

A second real STT call checks that the synthesized audio retained the intended meaning. This closes the voice round trip.


In [ ]:
roundtrip = agent.transcribe(
    reply_path, model=values["PRAVAL_OPENAI_TRANSCRIPTION_MODEL"],
    language="en", timeout=60,
)
assert expected_reply <= normalized_words(roundtrip)
stage("OpenAI STT", "round-trip verified", roundtrip)
show_json(
    {
        "input_transcript": transcript,
        "agent_reply": response.content,
        "roundtrip_transcript": roundtrip,
        "input_wav": input_info, "reply_wav": reply_info,
    },
    "Real STT → Agent → TTS → STT evidence",
)


### What just happened?

The generated audio was understandable to an independent transcription request and preserved the required semantic terms.

### Why this matters

Round-trip verification catches empty, corrupt, or semantically wrong synthesized audio.


### Send multimodal image input

The same Agent API accepts typed text and image ContentParts. A committed red PNG makes the expected visual answer deterministic while the provider call remains real.


In [ ]:
image_fixture = find_example_asset("certification/assets/image_input.png.base64")
image_bytes = base64.b64decode(image_fixture.read_text())
image_path = output / "red.png"
image_path.write_bytes(image_bytes)
display(Image(filename=str(image_path), width=96, height=96))

image_response = agent.generate([
    ContentPart.text_part("Name the dominant color in this image."),
    ContentPart.image_base64(
        base64.b64encode(image_bytes).decode("ascii"), "image/png"
    ),
])
assert "red" in image_response.content.lower()
stage("multimodal model", "image inspected", image_response.content)
show_json({"answer": image_response.content}, "Multimodal response")
show_timeline()


### What just happened?

ModelRuntime validated image capability, serialized typed content, and normalized the provider's visual answer.

### Why this matters

Typed content parts keep text, images, files, audio, and video distinct at the runtime boundary.


## Your turn

Change the requested reply to another short phrase, then update the semantic word assertion. Keep the output bounded and continue to validate the generated WAV.


In [ ]:
# requested = "Praval agents exchange structured knowledge."
# Generate the reply, synthesize it, transcribe it, and assert key words from requested.


## Common mistake

**Mistake:** Calling request-based STT/TTS a realtime voice session.

**Correction:** Use `transcribe()` and `speak()` for discrete requests. A realtime session needs a persistent bidirectional connection and continuous event handling, which is deferred.


<details>
<summary>Under the hood</summary>

Transcription and speech are provider-neutral ModelRuntime requests with typed audio responses. Multimodal content uses ContentPart validation against provider capabilities. Praval 0.8 does not add a persistent realtime session abstraction; it supports bounded request/response media operations.

</details>


## Recap

- The voice gate uses real provider STT and TTS in both directions.
- WAV structure and semantic content are independently verified.
- Multimodal content travels through typed provider-neutral parts.
- Request-based media is distinct from persistent realtime sessions.


### Next lesson

You have completed the course sequence. Continue with the case studies to see these capabilities combined into complete systems with fewer line-by-line explanations.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
agent.close()
workspace.cleanup()
stage("agent", "closed", "temporary media removed")
